# RT-DETR Final Layout — Garbage & Bin Detection

End-to-end pipeline: data preparation → RT-DETR-L training → evaluation → qualitative analysis → VLM-powered per-detection description.

**Pipeline overview:**
1. Environment setup
2. Data preparation & 85/15 re-split
3. Model training
4. Evaluation
5. Qualitative visualisation
6. Trial set inference
7. VLM integration (Gemini / InternVL2)


## 1. Environment Setup

Install required packages. Run this cell first on a fresh Colab runtime.

In [ ]:
!pip install -q transformers==4.44.2 accelerate timm einops
!pip install -q ultralytics
import ultralytics
ultralytics.checks()

## 2. Data Preparation

Imports, path config, 85/15 re-split of the merged dataset, and YAML generation. **Edit `SAVE_DIR` and `MERGED` before running.**

In [ ]:
import os, shutil, random, yaml
from pathlib import Path
from ultralytics import RTDETR
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

# ── EDIT THESE ────────────────────────────────────────────────────────────────
SAVE_DIR = '/content/drive/MyDrive/new/garbage_results'
MERGED   = Path('/content/drive/MyDrive/new/dataset_merged')
# ──────────────────────────────────────────────────────────────────────────────

os.makedirs(SAVE_DIR, exist_ok=True)

CLASS_NAMES = ['garbage', 'bin']
COLORS      = ['#e74c3c', '#3498db']

# ── Gather ALL images (train + val) and re-split 85/15 ───────────────────────
all_imgs = sorted(list((MERGED / 'train' / 'images').iterdir()))

random.seed(42)
random.shuffle(all_imgs)

n_train = int(len(all_imgs) * 0.85)
train_imgs = all_imgs[:n_train]
val_imgs   = all_imgs[n_train:]

RESPLIT = Path('/content/dataset_resplit')
for split in ['train', 'val']:
    (RESPLIT / split / 'images').mkdir(parents=True, exist_ok=True)
    (RESPLIT / split / 'labels').mkdir(parents=True, exist_ok=True)

def find_label(img_path):
    for split in ['train', 'val']:
        lbl = MERGED / split / 'labels' / (img_path.stem + '.txt')
        if lbl.exists():
            return lbl
    return None

for img_path in train_imgs:
    shutil.copy(img_path, RESPLIT / 'train' / 'images' / img_path.name)
    lbl = find_label(img_path)
    if lbl:
        shutil.copy(lbl, RESPLIT / 'train' / 'labels' / (img_path.stem + '.txt'))

for img_path in val_imgs:
    shutil.copy(img_path, RESPLIT / 'val' / 'images' / img_path.name)
    lbl = find_label(img_path)
    if lbl:
        shutil.copy(lbl, RESPLIT / 'val' / 'labels' / (img_path.stem + '.txt'))

print(f'Re-split: train={len(train_imgs)}  val={len(val_imgs)}')

# ── YAML ──────────────────────────────────────────────────────────────────────
MERGED_YAML = '/content/dataset_merged.yaml'
with open(MERGED_YAML, 'w') as f:
    yaml.dump({
        'path'  : str(RESPLIT),
        'train' : 'train/images',
        'val'   : 'val/images',
        'nc'    : 2,
        'names' : CLASS_NAMES
    }, f)
print('YAML ready:', MERGED_YAML)

## 3. Training

Trains RT-DETR-L on the re-split dataset for 100 epochs and saves the run to Drive.

In [ ]:
# ── Train ─────────────────────────────────────────────────────────────────────
model = RTDETR('rtdetr-l.pt')

model.train(
    data     = MERGED_YAML,
    epochs   = 100,
    imgsz    = 640,
    batch    = 8,
    device   = 0,
    project  = '/content/runs',
    name     = 'rtdetr_l_merged',
    patience = 15,
    verbose  = True
)

# ── Copy to Drive ─────────────────────────────────────────────────────────────
LOCAL_RUN = Path('/content/runs/rtdetr_l_merged')
DRIVE_RUN = Path(f'{SAVE_DIR}/runs/rtdetr_l_merged')
shutil.copytree(LOCAL_RUN, DRIVE_RUN, dirs_exist_ok=True)
print(f'Weights saved to: {DRIVE_RUN}/weights/best.pt')

## 4. Evaluation

Computes Precision, Recall, mAP@0.50, mAP@0.5:95, and F1 on the validation split.

In [ ]:
# ── Evaluate ──────────────────────────────────────────────────────────────────
best_rtdetr = RTDETR('/content/runs/rtdetr_l_merged/weights/best.pt')

rt_metrics = best_rtdetr.val(
    data    = MERGED_YAML,
    split   = 'val',
    device  = 0,
    verbose = True,
    conf    = 0.25,
    iou     = 0.5,
)

rt_map50   = float(rt_metrics.box.map50)
rt_map5095 = float(rt_metrics.box.map)
rt_p       = float(rt_metrics.box.mp)
rt_r       = float(rt_metrics.box.mr)

print(f'\n── RT-DETR-l Results ──')
print(f'  Precision  : {rt_p:.4f}')
print(f'  Recall     : {rt_r:.4f}')
print(f'  mAP@0.50   : {rt_map50:.4f}')
print(f'  mAP@0.5:95 : {rt_map5095:.4f}')
print(f'  F1 Score   : {2*rt_p*rt_r/(rt_p+rt_r+1e-8):.4f}')

## 5. Qualitative Visualisation — Validation Set

Renders bounding-box predictions on all validation images.

In [ ]:
# ── Visualise predictions on val images ───────────────────────────────────────
val_images = sorted(list((RESPLIT / 'val' / 'images').iterdir()))

cols = 3
rows = max(1, -(-len(val_images) // cols))
fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 5 * rows))
axes = np.array(axes).flatten()

for idx, img_path in enumerate(val_images):
    ax      = axes[idx]
    img     = Image.open(img_path).convert('RGB')
    results = best_rtdetr.predict(str(img_path), conf=0.25, device=0, verbose=False)[0]

    ax.imshow(img)
    for box in results.boxes:
        cls_id      = int(box.cls)
        conf        = float(box.conf)
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        color = COLORS[cls_id % len(COLORS)]
        rect  = patches.FancyBboxPatch(
            (x1, y1), x2-x1, y2-y1,
            linewidth=2, edgecolor=color, facecolor='none', boxstyle='round,pad=2'
        )
        ax.add_patch(rect)
        ax.text(x1, y1-6, f'{CLASS_NAMES[cls_id]} {conf:.0%}',
                color='white', fontsize=9, fontweight='bold',
                bbox=dict(facecolor=color, edgecolor='none', pad=2, alpha=0.85))

    n_det = len(results.boxes)
    ax.set_title(f'{img_path.name} — {n_det} detection{"s" if n_det != 1 else ""}', fontsize=9)
    ax.axis('off')

for idx in range(len(val_images), len(axes)):
    axes[idx].axis('off')

plt.suptitle('RT-DETR-l — Val set predictions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/rtdetr_val_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to /content/rtdetr_val_predictions.png')

## 6. Trial Set Inference

Runs inference on unseen trial images. Two source directories are provided below — use the one that matches your Drive layout. Edit `NEWDATA_DIR` and `WEIGHTS`.

### 6a. `TrialDatasetChosen` directory

In [ ]:
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
from ultralytics import YOLO

# ── EDIT THESE ────────────────────────────────────────────────────────────────
WEIGHTS     = '/content/runs/rtdetr_l_merged/weights/best.pt'
NEWDATA_DIR = Path('/content/drive/MyDrive/TrialDatasetChosen')
CONF_THRESH = 0.25
# ──────────────────────────────────────────────────────────────────────────────

model       = YOLO(WEIGHTS)
CLASS_NAMES = ['garbage', 'bin']
COLORS      = ['#e74c3c', '#3498db']

test_images = sorted([f for f in NEWDATA_DIR.iterdir()
                       if f.suffix.lower() in ('.jpg', '.jpeg', '.png')])
print(f'Found {len(test_images)} images in {NEWDATA_DIR}')

cols = 3
rows = max(1, -(-len(test_images) // cols))
fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 5 * rows))
axes = np.array(axes).flatten()

for idx, img_path in enumerate(test_images):
    ax  = axes[idx]
    img = Image.open(img_path).convert('RGB')

    results = model.predict(str(img_path), conf=CONF_THRESH, device='cpu', verbose=False)[0]

    ax.imshow(img)
    for box in results.boxes:
        cls_id          = int(box.cls)
        conf            = float(box.conf)
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        color           = COLORS[cls_id % len(COLORS)]
        ax.add_patch(patches.FancyBboxPatch(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=2, edgecolor=color, facecolor='none', boxstyle='round,pad=2'
        ))
        ax.text(x1, y1 - 6, f'{CLASS_NAMES[cls_id]} {conf:.0%}',
                color='white', fontsize=9, fontweight='bold',
                bbox=dict(facecolor=color, edgecolor='none', pad=2, alpha=0.85))

    n_det = len(results.boxes)
    ax.set_title(f'{img_path.name} — {n_det} detection{"s" if n_det != 1 else ""}', fontsize=9)
    ax.axis('off')

for idx in range(len(test_images), len(axes)):
    axes[idx].axis('off')

plt.suptitle('RT-DETR-l — Trial predictions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/trial_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved to Drive')

### 6b. `Trial` directory

In [ ]:
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
from ultralytics import YOLO

# ── EDIT THESE ────────────────────────────────────────────────────────────────
WEIGHTS     = '/content/runs/rtdetr_l_merged/weights/best.pt'
NEWDATA_DIR = Path('/content/drive/MyDrive/Trial')
CONF_THRESH = 0.25
# ──────────────────────────────────────────────────────────────────────────────

model       = YOLO(WEIGHTS)
CLASS_NAMES = ['garbage', 'bin']
COLORS      = ['#e74c3c', '#3498db']

test_images = sorted([f for f in NEWDATA_DIR.iterdir()
                       if f.suffix.lower() in ('.jpg', '.jpeg', '.png')])
print(f'Found {len(test_images)} images in {NEWDATA_DIR}')

cols = 3
rows = max(1, -(-len(test_images) // cols))
fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 5 * rows))
axes = np.array(axes).flatten()

for idx, img_path in enumerate(test_images):
    ax  = axes[idx]
    img = Image.open(img_path).convert('RGB')

    results = model.predict(str(img_path), conf=CONF_THRESH, device='cpu', verbose=False)[0]

    ax.imshow(img)
    for box in results.boxes:
        cls_id          = int(box.cls)
        conf            = float(box.conf)
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        color           = COLORS[cls_id % len(COLORS)]
        ax.add_patch(patches.FancyBboxPatch(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=2, edgecolor=color, facecolor='none', boxstyle='round,pad=2'
        ))
        ax.text(x1, y1 - 6, f'{CLASS_NAMES[cls_id]} {conf:.0%}',
                color='white', fontsize=9, fontweight='bold',
                bbox=dict(facecolor=color, edgecolor='none', pad=2, alpha=0.85))

    n_det = len(results.boxes)
    ax.set_title(f'{img_path.name} — {n_det} detection{"s" if n_det != 1 else ""}', fontsize=9)
    ax.axis('off')

for idx in range(len(test_images), len(axes)):
    axes[idx].axis('off')

plt.suptitle('RT-DETR-l — Trial predictions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/trial_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved to Drive')

## 7. VLM Integration

Pairs RT-DETR detections with a Vision-Language Model to describe each crop. Two VLM backends are provided — choose one.

> **Note:** Configure your API key via Colab Secrets (`userdata.get`) rather than hard-coding it in the cell.

### 7a. Gemini backend

In [ ]:
import os, shutil, random, time, textwrap
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
from ultralytics import RTDETR
import google.generativeai as genai
from google.colab import userdata

# ── EDIT THESE ────────────────────────────────────────────────────────────────
BEST_PT      = '/content/runs/rtdetr_l_merged/weights/best.pt'
SAVE_DIR     = '/content/drive/MyDrive/new/garbage_results'
RESPLIT      = Path('/content/dataset_resplit')
N_SAMPLES    = 15
CONF_THRESH  = 0.25
# ──────────────────────────────────────────────────────────────────────────────

# ── 1. Save best.pt to Drive ──────────────────────────────────────────────────
os.makedirs(SAVE_DIR, exist_ok=True)
drive_weights = f'{SAVE_DIR}/rtdetr_l_best.pt'
shutil.copy(BEST_PT, drive_weights)
print(f'✅ best.pt saved to Drive: {drive_weights}')

# ── 2. Load model ─────────────────────────────────────────────────────────────
model = RTDETR(BEST_PT)

# ── 3. Sample up to 15 val images ────────────────────────────────────────────
all_val = sorted((RESPLIT / 'val' / 'images').iterdir())
random.seed(42)
sample  = random.sample(all_val, min(N_SAMPLES, len(all_val)))
print(f'Sampled {len(sample)} images from val set')

# ── 4. Gemini VLM setup ───────────────────────────────────────────────────────
genai.configure(api_key="...")
vlm = genai.GenerativeModel('models/gemma-4-31b-it')

PROMPTS = {
    'garbage': textwrap.dedent("""
        Look at this cropped image of garbage.
        1. Classify quantity as exactly one of: "a little" | "a lot"
        2. Write one short sentence describing what you see.
        Reply in this exact format:
        QUANTITY: <a little or a lot>
        DESCRIPTION: <one sentence>
    """).strip(),

    'bin': textwrap.dedent("""
        Look at this cropped image of a garbage bin.
        1. Classify fill level as exactly one of: "empty" | "mid full" | "full"
        2. Write one short sentence describing what you see.
        Reply in this exact format:
        LEVEL: <empty or mid full or full>
        DESCRIPTION: <one sentence>
    """).strip(),
}

def call_vlm(crop: Image.Image, class_name: str, retries=3):
    prompt = PROMPTS[class_name]
    for attempt in range(retries):
        try:
            response = vlm.generate_content([prompt, crop])
            raw      = response.text.strip()

            # Parse label
            label, desc = 'unknown', ''
            for line in raw.splitlines():
                line = line.strip()
                if line.upper().startswith('QUANTITY:'):
                    label = line.split(':', 1)[1].strip()
                elif line.upper().startswith('LEVEL:'):
                    label = line.split(':', 1)[1].strip()
                elif line.upper().startswith('DESCRIPTION:'):
                    desc = line.split(':', 1)[1].strip()
            return label, desc

        except Exception as e:
            wait = (attempt + 1) * 5
            if '503' in str(e) or '429' in str(e):
                print(f'    ⏳ Rate limit, retrying in {wait}s...')
                time.sleep(wait)
            else:
                print(f'    ⚠️  VLM error: {e}')
                break
    return 'error', ''

# ── 5. Detection + VLM loop ───────────────────────────────────────────────────
CLASS_NAMES = ['garbage', 'bin']
BOX_COLORS  = ['#e74c3c', '#3498db']   # red=garbage, blue=bin
VLM_COLOR   = '#8e44ad'                # purple for VLM label

cols = 3
rows = max(1, -(-len(sample) // cols))
fig, axes = plt.subplots(rows, cols, figsize=(7 * cols, 6 * rows))
axes = np.array(axes).flatten()

for idx, img_path in enumerate(sample):
    ax      = axes[idx]
    img     = Image.open(img_path).convert('RGB')
    W, H    = img.size

    print(f'[{idx+1}/{len(sample)}] {img_path.name}', end='  →  ')
    results = model.predict(str(img_path), conf=CONF_THRESH, device=0, verbose=False)[0]
    print(f'{len(results.boxes)} detection(s)')

    ax.imshow(img)

    for box in results.boxes:
        cls_id          = int(box.cls)
        det_conf        = float(box.conf)
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        class_name      = CLASS_NAMES[cls_id]
        box_color       = BOX_COLORS[cls_id]

        # Crop with padding
        pad  = 8
        crop = img.crop((max(0, x1-pad), max(0, y1-pad),
                         min(W, x2+pad), min(H, y2+pad)))

        vlm_label, vlm_desc = call_vlm(crop, class_name)
        print(f'      [{class_name}] → {vlm_label} | {vlm_desc}')

        # Bounding box
        ax.add_patch(patches.FancyBboxPatch(
            (x1, y1), x2-x1, y2-y1,
            linewidth=2.5, edgecolor=box_color, facecolor='none',
            boxstyle='round,pad=2'
        ))

        # Detection label — top of box
        ax.text(x1, y1-10, f'{class_name} {det_conf:.0%}',
                color='white', fontsize=8, fontweight='bold',
                bbox=dict(facecolor=box_color, edgecolor='none', pad=2, alpha=0.9))

        # VLM label — just below box
        ax.text(x1, y2+16, f'🧠 {vlm_label}',
                color='white', fontsize=8, fontweight='bold',
                bbox=dict(facecolor=VLM_COLOR, edgecolor='none', pad=2, alpha=0.9))

        # Description — below VLM label, wrapped
        wrapped = textwrap.fill(vlm_desc, width=40)
        ax.text(x1, y2+32, wrapped,
                color='black', fontsize=7, style='italic',
                verticalalignment='top')

        time.sleep(1.5)   # respect free-tier rate limit (15 RPM)

    ax.set_title(img_path.name, fontsize=8)
    ax.axis('off')

for idx in range(len(sample), len(axes)):
    axes[idx].axis('off')

plt.suptitle('RT-DETR-l + Gemini — Val Sample Analysis',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()

out_path = f'{SAVE_DIR}/val_sample_vlm.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'\n✅ Plot saved to Drive: {out_path}')

### 7b. InternVL2-4B backend

In [ ]:
# ── 0. Install correct versions first ─────────────────────────────────────────
import os, shutil, random, textwrap
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import torch
from ultralytics import RTDETR
from transformers import AutoTokenizer, AutoModel
from torchvision import transforms

# ── EDIT THESE ────────────────────────────────────────────────────────────────
BEST_PT     = '/content/runs/rtdetr_l_merged/weights/best.pt'
SAVE_DIR    = '/content/drive/MyDrive/new/garbage_results'
RESPLIT     = Path('/content/dataset_resplit')
N_SAMPLES   = 15
CONF_THRESH = 0.25
# ──────────────────────────────────────────────────────────────────────────────

# ── 1. Save best.pt to Drive ──────────────────────────────────────────────────
os.makedirs(SAVE_DIR, exist_ok=True)
shutil.copy(BEST_PT, f'{SAVE_DIR}/rtdetr_l_best.pt')
print('✅ best.pt saved to Drive')

# ── 2. Load RT-DETR ───────────────────────────────────────────────────────────
detector = RTDETR(BEST_PT)
print('✅ RT-DETR ready')

# ── 3. Load InternVL2-4B ──────────────────────────────────────────────────────
print('Loading InternVL2-4B (this takes ~1-2 min)...')
MODEL_NAME = 'OpenGVLab/InternVL2-4B'

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME, trust_remote_code=True, use_fast=False
)
internvl = AutoModel.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    trust_remote_code=True,
).to('cuda').eval()

print('✅ InternVL2-4B ready')

# ── 4. VLM helpers ────────────────────────────────────────────────────────────
def preprocess_crop(pil_img, image_size=448):
    transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])
    return transform(pil_img).unsqueeze(0).to(torch.float16).cuda()

PROMPTS = {
    'garbage': textwrap.dedent("""
        Look at this cropped image of garbage.
        1. Classify quantity as exactly one of: "a little" | "a lot" | "model wrong"
        2. Write one short sentence describing what you see.
        Reply in this exact format:
        QUANTITY: <a little or a lot or model wrong>
        DESCRIPTION: <one sentence>
    """).strip(),

    'bin': textwrap.dedent("""
        Look at this cropped image of a garbage bin.
        1. Classify fill level as exactly one of: "empty" | "mid full" | "full" | "model wrong"
        2. Write one short sentence describing what you see.
        Reply in this exact format:
        LEVEL: <empty or mid full or full or model wrong>
        DESCRIPTION: <one sentence>
    """).strip(),
}

def call_vlm(crop: Image.Image, class_name: str):
    try:
        pixel_values = preprocess_crop(crop)
        prompt       = f'<image>\n{PROMPTS[class_name]}'
        response     = internvl.chat(
            tokenizer         = tokenizer,
            pixel_values      = pixel_values,
            question          = prompt,
            generation_config = dict(max_new_tokens=80, do_sample=False)
        )
        raw        = response.strip()
        label, desc = 'unknown', ''
        for line in raw.splitlines():
            line = line.strip()
            if line.upper().startswith('QUANTITY:'):
                label = line.split(':', 1)[1].strip()
            elif line.upper().startswith('LEVEL:'):
                label = line.split(':', 1)[1].strip()
            elif line.upper().startswith('DESCRIPTION:'):
                desc = line.split(':', 1)[1].strip()
        return label, desc
    except Exception as e:
        print(f'    ⚠️  VLM error: {e}')
        return 'error', ''

# ── 5. Sample val images ──────────────────────────────────────────────────────
all_val = sorted((RESPLIT / 'val' / 'images').iterdir())
random.seed(42)
sample  = random.sample(all_val, min(N_SAMPLES, len(all_val)))
print(f'Sampled {len(sample)} images from val set')

# ── 6. Detection + VLM loop ───────────────────────────────────────────────────
CLASS_NAMES = ['garbage', 'bin']
BOX_COLORS  = ['#e74c3c', '#3498db']   # red=garbage, blue=bin
VLM_COLOR   = '#27ae60'                # green for InternVL

cols = 3
rows = max(1, -(-len(sample) // cols))
fig, axes = plt.subplots(rows, cols, figsize=(7 * cols, 6 * rows))
axes = np.array(axes).flatten()

for idx, img_path in enumerate(sample):
    ax   = axes[idx]
    img  = Image.open(img_path).convert('RGB')
    W, H = img.size

    print(f'[{idx+1}/{len(sample)}] {img_path.name}', end='  →  ')
    results = detector.predict(str(img_path), conf=CONF_THRESH, device=0, verbose=False)[0]
    print(f'{len(results.boxes)} detection(s)')

    ax.imshow(img)

    for box in results.boxes:
        cls_id          = int(box.cls)
        det_conf        = float(box.conf)
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        class_name      = CLASS_NAMES[cls_id]
        box_color       = BOX_COLORS[cls_id]

        pad  = 8
        crop = img.crop((max(0, x1-pad), max(0, y1-pad),
                         min(W, x2+pad), min(H, y2+pad)))

        vlm_label, vlm_desc = call_vlm(crop, class_name)
        print(f'      [{class_name}] → {vlm_label} | {vlm_desc}')

        # Bounding box
        ax.add_patch(patches.FancyBboxPatch(
            (x1, y1), x2-x1, y2-y1,
            linewidth=2.5, edgecolor=box_color, facecolor='none',
            boxstyle='round,pad=2'
        ))
        # Detection label — top of box
        ax.text(x1, y1-10, f'{class_name} {det_conf:.0%}',
                color='white', fontsize=8, fontweight='bold',
                bbox=dict(facecolor=box_color, edgecolor='none', pad=2, alpha=0.9))
        # VLM label — below box
        ax.text(x1, y2+16, f'🧠 {vlm_label}',
                color='white', fontsize=8, fontweight='bold',
                bbox=dict(facecolor=VLM_COLOR, edgecolor='none', pad=2, alpha=0.9))
        # Description
        wrapped = textwrap.fill(vlm_desc, width=40)
        ax.text(x1, y2+32, wrapped,
                color='black', fontsize=7, style='italic',
                verticalalignment='top')

    ax.set_title(img_path.name, fontsize=8)
    ax.axis('off')

for idx in range(len(sample), len(axes)):
    axes[idx].axis('off')

plt.suptitle('RT-DETR-l + InternVL2-4B — Val Sample Analysis',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()

out_path = f'{SAVE_DIR}/val_sample_internvl.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'\n✅ Plot saved to Drive: {out_path}')